# layerbn: interactive analysis

`layerbn run settings.yml` runs everything and writes a report. This notebook runs
the same steps one at a time, so you can look at intermediate results. It needs
no code changes: the variables, layers and settings come from `variables.xlsx`
and `settings.yml`.

Run the cells in order. The kernel must be the Python environment in which
layerbn is installed.

In [ ]:
import warnings

from layerbn import figures
from layerbn.analysis import Analysis

warnings.filterwarnings("ignore")   # pyAgrum and scikit-learn print notices you cannot act on

study = Analysis.from_files("settings.yml")   # reads settings.yml, variables.xlsx and the data
print(study.summary())

## Layers and variables

Check that the layers are in the order of influence you intend. Arcs run from a layer
to itself or to a later layer, and from top to bottom within a column.

In [ ]:
figures.design_figure(study.project)

## Bins

Every result depends on how continuous variables were binned. Check the edges.

In [ ]:
study.bins()

In [ ]:
figures.plot_bins(study.df, study.design, study.template(study.project.network().name))

## Network and arc stability

The first network in `settings.yml`. Change `NETWORK` to look at another one.
The bootstrap is the slow step: set a small `bootstrap: resamples:` while trying things out.

In [ ]:
NETWORK = study.project.network().name

figures.network_figure(study, NETWORK, stability=True)

In [ ]:
edges = study.stable_edges(NETWORK)
edges.head(25)

In [ ]:
figures.plot_stability(edges)

To read the neighbourhood of one variable, fade everything else:
`figures.network_figure(study, NETWORK, highlight=["VARIABLE"])`.

## Which variables tell most about the targets

In [ ]:
importance = study.importance(NETWORK)
importance.head(20)

In [ ]:
for target in importance["Target"].unique():
    display(figures.plot_importance(importance, target))

## Profiles and sweep

Only available if `analyses: profiles:` and `analyses: sweep:` are in `settings.yml`.

In [ ]:
profiles = study.profiles()
figures.plot_profiles(profiles)

In [ ]:
sweep, meta = study.sweep()
figures.plot_sweep(sweep, meta)

## Save a figure

`figures.save_figure(fig, "results/my_figure")` writes PNG and PDF.